# 03 - Task-Driven Teams with CrewAI

## Scenario: Northstar Incident Response Team

While AutoGen focuses on conversational interactions between agents, **CrewAI** is heavily task-driven. In CrewAI, you define specific `Tasks` and assign them to `Agents`. This is highly effective when your workflow is a strict assembly line rather than a free-flowing chat.

In [ ]:
import os
from crewai import Agent, Task, Crew, Process

os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "dummy-key")

# 1. Define Agents with clear roles and backstories
investigator = Agent(
    role='Senior SRE Investigator',
    goal='Identify the root cause of service disruptions.',
    backstory='You are a veteran Site Reliability Engineer at Northstar. You trust logs over intuition.',
    verbose=True,
    allow_delegation=False
)

reviewer = Agent(
    role='Security & Compliance Reviewer',
    goal='Ensure mitigations do not violate compliance policies.',
    backstory='You are a strict compliance officer. You reject any mitigation that involves touching production DBs directly.',
    verbose=True,
    allow_delegation=False
)


## 1. Defining Tasks

Unlike AutoGen where agents just chat, CrewAI tasks have expected outputs and are explicitly assigned to agents.

In [ ]:
# 2. Define Tasks
investigation_task = Task(
    description='Analyze the EU checkout failure incident based on recent logs.',
    expected_output='A 3-bullet point summary of the root cause.',
    agent=investigator
)

review_task = Task(
    description='Review the root cause and propose a safe mitigation plan.',
    expected_output='A mitigation plan that is compliant with Northstar policies.',
    agent=reviewer
)


## 2. Assembling the Crew

A Crew manages the execution of tasks. A sequential process means tasks run one after another.

In [ ]:
# 3. Assemble the Crew
incident_crew = Crew(
    agents=[investigator, reviewer],
    tasks=[investigation_task, review_task],
    process=Process.sequential, # Tasks run in order
    verbose=True
)

# 4. Kickoff the crew
try:
    result = incident_crew.kickoff()
    print("Crew finished with result:\n")
    print(result)
except Exception as e:
    print(f"API Error (Expected if using a dummy key): {e}")


## Watch For

- **Over-delegation**: Agents can delegate tasks to each other indefinitely if `allow_delegation=True` is used without boundaries.
- **Ignoring Expected Output**: LLMs might ignore the `expected_output` format if it's too complex. Use Pydantic models for output if necessary.
- **Task Order**: In sequential processes, if the first task fails, the entire crew might produce garbage.

## Checkpoint

**1. How does CrewAI's design philosophy differ from AutoGen's?**
- A) CrewAI is only for Python 2.
- B) CrewAI is conversation-driven, while AutoGen is task-driven.
- C) CrewAI is task-driven (agents execute specific assigned tasks), while AutoGen is conversation-driven (agents chat with each other).
- D) They are exactly the same.

**2. What happens in a `Process.sequential` crew?**
- A) All tasks run in parallel.
- B) The output of Task 1 is automatically passed as context to Task 2.
- C) The agents vote on which task to do first.
- D) The crew is deleted after running.
